# Dynamic Frequency-Biased HNSW with Spatial Node Heterogeneity (SIFT 1M)

## Overview & Theoretical Framework

Hierarchical Navigable Small World (HNSW) graphs construct a multi-layer index where lower layers contain fine-grained nearest neighbor connections and upper layers act as coarse skip-lists for fast greedy routing. In standard HNSW, nodes are assigned to layers according to a random exponential probability distribution:

$$P(\text{level} \ge l) = \left(\frac{1}{M}\right)^l$$

However, real-world query workloads are rarely uniform. Under non-stationary or power-law query streams, certain vectors are queried far more frequently than others. Standard HNSW does not adapt its graph topology to search frequency patterns.

To optimize HNSW search latency and recall under frequency-skewed workloads, this notebook implements **Dynamic HNSW** with two advanced architectural extensions:

---

### 1. Probabilistic Frequency-Biased Skip List
Instead of a purely uniform random level assignment, vector access frequencies $f_i$ are leveraged to bias the probability of a node ascending to higher HNSW layers. 

For a node $i$ with query access frequency $f_i$ (normalized to $\bar{f}_i \in [0, 1]$), the effective level distribution parameter $mL_i$ is scaled probabilistically:

$$mL_i = mL \cdot (1 + \alpha \cdot \bar{f}_i), \quad \text{where } mL = \frac{1}{\ln M}$$

$$l_i = \lfloor -\ln(r_i) \cdot mL_i \rfloor, \quad r_i \sim U(0, 1]$$

- **Key Property**: Nodes with high query frequencies have a **significantly higher probability** of being promoted to top layers ($l \ge 1$).
- **Stochasticity Preserved**: Promotion remains probabilistic, preserving the small-world logarithmic graph properties.

---

### 2. Node Heterogeneity (Spatial Diversity Preservation)

#### The Local Minimum / Spatial Trapline Problem
If a search workload is highly localized in a specific region of the vector space, pure frequency biasing would promote *only* nodes residing in that small cluster to upper HNSW layers. Consequently, top-level entry points become tightly grouped in one region, creating a "spatial trap" (local minimum) for queries targeted at other parts of the vector space.

To prevent this spatial clustering, node promotion to higher layers is constrained by **Spatial Heterogeneity**:

#### Approach A: Min-Distance Thresholding (Max-Min Diversity)
When evaluating candidate nodes for layer $l$ (sorted by query frequency), a candidate node $i$ is accepted into layer $l$ only if its squared distance to all already accepted nodes $S_l$ in layer $l$ exceeds a spatial distance threshold $\tau$:

$$d_{\min}(i, S_l) = \min_{j \in S_l} \|x_i - x_j\|^2 \ge \tau$$

- **Effect**: Guarantees that promoted high-frequency nodes maintain a minimum spatial distance, spreading upper-layer entry points evenly across the vector space.

#### Approach B: Layer SSE Gain (Sum of Squared Errors / Coverage)
Alternatively, upper-layer candidate nodes are evaluated based on their contribution to reducing the quantization Sum of Squared Errors (SSE) / coverage loss over the vector space. For candidate $i$ and existing layer $l$ nodes $S_l$:

$$\text{SSE}(S_l) = \sum_{v \in \mathcal{D}} \min_{j \in S_l} \|x_v - x_j\|^2$$

$$\Delta \text{SSE}(i) = \frac{\text{SSE}(S_l) - \text{SSE}(S_l \cup \{i\})}{\text{SSE}(S_l)} \ge \epsilon$$

- **Effect**: Accepts candidate $i$ only if it provides a substantial relative reduction in quantization error (i.e., covers an under-represented spatial region).

---

### 3. Dataset: SIFT 1M
All experiments are conducted on the **SIFT 1M dataset** (`sift/sift_base.fvecs`), consisting of 1,000,000 128-dimensional SIFT descriptors.

---

### Notebook Structure
1. **Setup & Data Loading**: Reads `sift/sift_base.fvecs` and initializes Power-Law synthetic query generators.
2. **Base Class `InstrumentedHNSW`**: Baseline HNSW tracking hop counts and custom entry points.
3. **Class `DynamicBiasedHNSW`**: Implements probabilistic frequency biasing, Min-Distance heterogeneity, and Layer SSE Gain heterogeneity.
4. **Evaluation & Benchmarking**: Measures Recall@1 vs. Ground Truth exact 1-NN and average search hops.
5. **Visualizations**: Performance comparison charts and 2D PCA spatial projection of upper-layer node distributions.


In [ ]:
# Imports and Environment Setup
import os
import time
import heapq
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

# Set random seed for reproducibility
np.random.seed(42)
print("Libraries loaded successfully.")


In [ ]:
# Data Loading (SIFT 1M) and Synthetic Query Generation

def read_fvecs(filename, max_vectors=None):
    """Reads a .fvecs binary file into a 2D float32 numpy array."""
    data = np.fromfile(filename, dtype='int32')
    dim = data[0]
    total_vecs = len(data) // (dim + 1)
    if max_vectors is not None:
        total_vecs = min(total_vecs, max_vectors)
    reshaped = data[:total_vecs * (dim + 1)].reshape(-1, dim + 1)[:, 1:]
    return reshaped.view('float32').copy()

def generate_queries(data, n_queries=500, exponent=3.0, constant=2.0, n_clusters=3, random_state=42):
    """
    Generates synthetic query vectors following a power-law frequency distribution
    around n_clusters cluster centers in the vector space.
    """
    orig_dtype = data.dtype
    data_int = np.round(data).astype(np.int64) if np.issubdtype(orig_dtype, np.floating) else data.astype(np.int64)
    rng = np.random.default_rng(random_state)
    _, d = data_int.shape

    L = data_int.min(axis=0)
    U = data_int.max(axis=0)

    centers = rng.integers(L, U + 1, size=(n_clusters, d))
    assignments = rng.integers(0, n_clusters, size=n_queries)
    queries = np.empty((n_queries, d), dtype=orig_dtype)

    for k in range(n_clusters):
        mask = assignments == k
        count = np.sum(mask)
        if count == 0:
            continue

        for j in range(d):
            values = np.arange(L[j], U[j] + 1, dtype=np.int64)
            weights = (np.abs(values - centers[k, j]) + constant) ** (-float(exponent))
            probs = weights / weights.sum()
            samples = rng.choice(values, size=count, p=probs)
            queries[mask, j] = samples.astype(orig_dtype)

    return queries

# Load SIFT 1M dataset (N_DATA can be set to None to load all 1,000,000 vectors)
sift_file = "sift/sift_base.fvecs"
if os.path.exists(sift_file):
    N_DATA = 5000  # Set to None for full 1M dataset benchmarking
    data = read_fvecs(sift_file, max_vectors=N_DATA)
    print(f"Loaded SIFT 1M subset with shape: {data.shape}")
else:
    print(f"Warning: {sift_file} not found. Please ensure SIFT 1M is downloaded.")


In [ ]:
# Base HNSW Implementation with Search Hop Instrumentation

class InstrumentedHNSW:
    """Base HNSW index supporting distance metric and search hop tracking."""
    
    def __init__(self, data: np.ndarray, M: int = 16, ef_construction: int = 40, random_state: int = 42, build_on_init: bool = True):
        self.data = data
        self.N, self.dim = data.shape
        self.M = M
        self.M0 = 2 * M
        self.ef_construction = ef_construction
        self.mL = 1.0 / np.log(M)
        self.rng = np.random.default_rng(random_state)

        self.layers: list[dict[int, list[int]]] = [{}]
        self.node_levels = np.zeros(self.N, dtype=np.int32)
        self.enter_point: int | None = None
        self.max_level: int = -1

        if build_on_init:
            self._build_index()

    def _dist_sq(self, q: np.ndarray, node_ids: list[int] | int) -> np.ndarray | float:
        """Vectorized squared Euclidean distance."""
        if isinstance(node_ids, (int, np.integer)):
            diff = self.data[node_ids] - q
            return float(np.dot(diff, diff))
        diff = self.data[node_ids] - q
        return np.sum(diff * diff, axis=1)

    def _assign_level(self, node_id: int) -> int:
        """Standard random exponential level assignment."""
        r = self.rng.uniform(1e-9, 1.0)
        return int(-np.log(r) * self.mL)

    def _build_index(self):
        """Constructs multi-layer HNSW graph structure."""
        self.layers = [{}]
        self.max_level = -1
        self.enter_point = None

        for i in range(self.N):
            level = self.node_levels[i]
            while len(self.layers) <= level:
                self.layers.append({})

            if self.enter_point is None:
                for lc in range(level + 1):
                    self.layers[lc][i] = []
                self.enter_point = i
                self.max_level = level
                continue

            curr_ep = self.enter_point
            q_vec = self.data[i]

            # Greedy top-down search to insertion level
            for lc in range(self.max_level, level, -1):
                changed = True
                curr_dist = self._dist_sq(q_vec, curr_ep)
                while changed:
                    changed = False
                    neighbors = self.layers[lc].get(curr_ep, [])
                    if not neighbors:
                        break
                    dists = self._dist_sq(q_vec, neighbors)
                    min_idx = np.argmin(dists)
                    if dists[min_idx] < curr_dist:
                        curr_dist = dists[min_idx]
                        curr_ep = neighbors[min_idx]
                        changed = True

            # Insert connections from insertion level down to 0
            for lc in range(min(level, self.max_level), -1, -1):
                candidates = self._search_layer(q_vec, curr_ep, ef=self.ef_construction, lc=lc)
                m_max = self.M0 if lc == 0 else self.M

                cand_ids = [idx for _, idx in candidates]
                cand_dists = self._dist_sq(q_vec, cand_ids)
                best_neighbors = [
                    cand_ids[idx] for idx in np.argsort(cand_dists)[:m_max] if cand_ids[idx] != i
                ]

                self.layers[lc][i] = best_neighbors

                for neighbor in best_neighbors:
                    self.layers[lc].setdefault(neighbor, []).append(i)
                    if len(self.layers[lc][neighbor]) > m_max:
                        n_neighbors = self.layers[lc][neighbor]
                        d = self._dist_sq(self.data[neighbor], n_neighbors)
                        self.layers[lc][neighbor] = [
                            n_neighbors[idx] for idx in np.argsort(d)[:m_max]
                        ]

                curr_ep = candidates[0][1]

            if level > self.max_level:
                self.max_level = level
                self.enter_point = i

    def _search_layer(self, q: np.ndarray, ep: int, ef: int, lc: int) -> list[tuple[float, int]]:
        """Beam search within a single layer."""
        v_dist = self._dist_sq(q, ep)
        visited = {ep}
        candidates = [(v_dist, ep)]
        w = [(-v_dist, ep)]

        while candidates:
            c_dist, c_id = heapq.heappop(candidates)
            furthest_w_dist = -w[0][0]

            if c_dist > furthest_w_dist:
                break

            neighbors = self.layers[lc].get(c_id, [])
            for n in neighbors:
                if n not in visited:
                    visited.add(n)
                    n_dist = self._dist_sq(q, n)
                    if n_dist < furthest_w_dist or len(w) < ef:
                        heapq.heappush(candidates, (n_dist, n))
                        heapq.heappush(w, (-n_dist, n))
                        if len(w) > ef:
                            heapq.heappop(w)

        return sorted([(-d, idx) for d, idx in w])

    def search(self, query: np.ndarray, ef: int = 32, custom_entry_id: int | None = None, enter_at_layer_zero: bool = False) -> dict:
        """Executes 1-NN search while tracking total traversal hops."""
        hops = 0
        if custom_entry_id is None:
            curr_obj = self.enter_point
            curr_level = self.max_level
        else:
            curr_obj = custom_entry_id
            curr_level = 0 if enter_at_layer_zero else min(self.node_levels[custom_entry_id], self.max_level)

        curr_dist = self._dist_sq(query, curr_obj)

        for lc in range(curr_level, 0, -1):
            changed = True
            while changed:
                changed = False
                neighbors = self.layers[lc].get(curr_obj, [])
                if not neighbors:
                    break
                dists = self._dist_sq(query, neighbors)
                min_idx = np.argmin(dists)
                if dists[min_idx] < curr_dist:
                    curr_dist = dists[min_idx]
                    curr_obj = neighbors[min_idx]
                    hops += 1
                    changed = True

        visited = {curr_obj}
        candidates = [(curr_dist, curr_obj)]
        w = [(-curr_dist, curr_obj)]

        while candidates:
            c_dist, c_id = heapq.heappop(candidates)
            furthest_w = -w[0][0]

            if c_dist > furthest_w:
                break

            hops += 1

            neighbors = self.layers[0].get(c_id, [])
            for n in neighbors:
                if n not in visited:
                    visited.add(n)
                    n_dist = self._dist_sq(query, n)
                    if n_dist < furthest_w or len(w) < ef:
                        heapq.heappush(candidates, (n_dist, n))
                        heapq.heappush(w, (-n_dist, n))
                        if len(w) > ef:
                            heapq.heappop(w)

        best_dist, best_node = min((-neg_d, idx) for neg_d, idx in w)
        return {"node": best_node, "distance_sq": best_dist, "hops": hops}


In [ ]:
# Dynamic Biased HNSW Class (Frequency Bias & Spatial Heterogeneity)

class DynamicBiasedHNSW(InstrumentedHNSW):
    """
    Advanced HNSW implementation supporting:
    1. Probabilistic Frequency-Biased Skip List
    2. Heterogeneity Approach A: Min-Distance Thresholding
    3. Heterogeneity Approach B: Layer SSE Gain
    """
    
    def __init__(self, data: np.ndarray, M: int = 16, ef_construction: int = 40, random_state: int = 42, build_on_init: bool = True):
        self.frequencies = np.zeros(len(data), dtype=np.float64)
        super().__init__(data, M=M, ef_construction=ef_construction, random_state=random_state, build_on_init=build_on_init)

    def assign_levels_biased(self, frequencies: dict[int, float] | np.ndarray, alpha: float = 3.0) -> np.ndarray:
        """
        Assigns levels probabilistically weighted by vector access frequency.
        Higher query frequency increases effective mL_i -> higher promotion probability.
        """
        if isinstance(frequencies, dict):
            freq_arr = np.zeros(self.N, dtype=np.float64)
            for idx, count in frequencies.items():
                if 0 <= idx < self.N:
                    freq_arr[idx] = count
        else:
            freq_arr = np.array(frequencies, dtype=np.float64)

        max_f = freq_arr.max()
        norm_freq = freq_arr / max_f if max_f > 0 else np.zeros_like(freq_arr)

        levels = np.zeros(self.N, dtype=np.int32)
        for i in range(self.N):
            r = self.rng.uniform(1e-9, 1.0)
            eff_mL = self.mL * (1.0 + alpha * norm_freq[i])
            levels[i] = int(-np.log(r) * eff_mL)

        return levels

    def rebuild_biased(self, frequencies: dict[int, float] | np.ndarray, alpha: float = 3.0):
        """Rebuilds HNSW graph using probabilistic frequency-biased level assignments."""
        self.node_levels = self.assign_levels_biased(frequencies, alpha=alpha)
        self._build_index()

    def rebuild_heterogeneous_min_dist(
        self,
        frequencies: dict[int, float] | np.ndarray,
        alpha: float = 3.0,
        min_dist_percentile: float = 20.0
    ):
        """
        Approach A: Probabilistic Frequency Bias + Min-Distance Spatial Heterogeneity.
        Promotes high-frequency candidates to layer l only if min distance to existing layer l nodes >= tau.
        """
        raw_levels = self.assign_levels_biased(frequencies, alpha=alpha)
        if isinstance(frequencies, dict):
            freq_arr = np.zeros(self.N, dtype=np.float64)
            for idx, count in frequencies.items():
                if 0 <= idx < self.N:
                    freq_arr[idx] = count
        else:
            freq_arr = np.array(frequencies, dtype=np.float64)

        # Estimate spatial distance scale threshold tau from dataset pairwise distances
        sample_ids = self.rng.choice(self.N, size=min(1000, self.N), replace=False)
        sample_vecs = self.data[sample_ids]
        pairwise_dists = np.sum((sample_vecs[:, None, :] - sample_vecs[None, :, :]) ** 2, axis=-1)
        min_dist_threshold = np.percentile(pairwise_dists[pairwise_dists > 0], min_dist_percentile)

        final_levels = np.zeros(self.N, dtype=np.int32)
        max_raw_level = int(raw_levels.max())

        final_levels[:] = 0

        for l in range(1, max_raw_level + 1):
            candidates = [i for i in range(self.N) if raw_levels[i] >= l]
            candidates.sort(key=lambda idx: freq_arr[idx], reverse=True)

            accepted_vecs = []
            for cand in candidates:
                cand_vec = self.data[cand]
                if not accepted_vecs:
                    accepted_vecs.append(cand_vec)
                    final_levels[cand] = l
                else:
                    acc_mat = np.array(accepted_vecs)
                    diff = acc_mat - cand_vec
                    min_d = np.min(np.sum(diff * diff, axis=1))
                    if min_d >= min_dist_threshold:
                        accepted_vecs.append(cand_vec)
                        final_levels[cand] = l

        self.node_levels = final_levels
        self._build_index()

    def rebuild_heterogeneous_sse(
        self,
        frequencies: dict[int, float] | np.ndarray,
        alpha: float = 3.0,
        min_sse_gain_ratio: float = 0.005
    ):
        """
        Approach B: Probabilistic Frequency Bias + Layer SSE Gain Heterogeneity.
        Promotes high-frequency candidates to layer l if adding the node yields relative SSE coverage gain >= epsilon.
        """
        raw_levels = self.assign_levels_biased(frequencies, alpha=alpha)
        if isinstance(frequencies, dict):
            freq_arr = np.zeros(self.N, dtype=np.float64)
            for idx, count in frequencies.items():
                if 0 <= idx < self.N:
                    freq_arr[idx] = count
        else:
            freq_arr = np.array(frequencies, dtype=np.float64)

        final_levels = np.zeros(self.N, dtype=np.int32)
        max_raw_level = int(raw_levels.max())

        final_levels[:] = 0

        # Sub-sample data to efficiently estimate SSE gain
        sub_ids = self.rng.choice(self.N, size=min(1000, self.N), replace=False)
        sub_data = self.data[sub_ids]

        for l in range(1, max_raw_level + 1):
            candidates = [i for i in range(self.N) if raw_levels[i] >= l]
            candidates.sort(key=lambda idx: freq_arr[idx], reverse=True)

            accepted_nodes = []
            prev_min_dists = None
            current_sse = None

            for cand in candidates:
                cand_dists = np.sum((sub_data - self.data[cand]) ** 2, axis=1)

                if not accepted_nodes:
                    accepted_nodes.append(cand)
                    final_levels[cand] = l
                    prev_min_dists = cand_dists.copy()
                    current_sse = float(np.sum(prev_min_dists))
                else:
                    new_min_dists = np.minimum(prev_min_dists, cand_dists)
                    new_sse = float(np.sum(new_min_dists))

                    sse_gain = current_sse - new_sse
                    relative_gain = sse_gain / max(current_sse, 1e-9)

                    if relative_gain >= min_sse_gain_ratio:
                        accepted_nodes.append(cand)
                        final_levels[cand] = l
                        prev_min_dists = new_min_dists
                        current_sse = new_sse

        self.node_levels = final_levels
        self._build_index()

print("DynamicBiasedHNSW class compiled successfully.")


In [ ]:
# Experimental Evaluation & Benchmarking Pipeline

def compute_ground_truth_1nn(data: np.ndarray, queries: np.ndarray) -> np.ndarray:
    """Computes exact 1-NN ground truth using vectorized Euclidean distance."""
    gt = np.empty(len(queries), dtype=np.int32)
    for i, q in enumerate(queries):
        diff = data - q
        dists = np.sum(diff * diff, axis=1)
        gt[i] = np.argmin(dists)
    return gt

# 1. Generate Power-Law query stream
queries = generate_queries(data, n_queries=300, exponent=3.0, constant=2.0, n_clusters=3, random_state=42)

# Split into warmup (to build frequency distribution) and evaluation queries
warmup_queries = queries[:150]
test_queries = queries[150:]

# Ground Truth for test set
print("Computing Ground Truth 1-NN...")
test_gt = compute_ground_truth_1nn(data, test_queries)

# 2. Initialize Base HNSW and collect vector access frequencies
index_base = DynamicBiasedHNSW(data, M=16, ef_construction=32, random_state=42, build_on_init=True)

frequencies = {}
for q in warmup_queries:
    res = index_base.search(q, ef=32)
    node = res['node']
    frequencies[node] = frequencies.get(node, 0) + 1

print(f"Collected frequencies across {len(warmup_queries)} warmup queries.")
print(f"Unique queried vectors: {len(frequencies)}, Max frequency: {max(frequencies.values())}")

# 3. Instantiate and build the 4 HNSW Index Variants
print("\nBuilding Index Variants...")

# Variant 1: Standard HNSW (Random level sampling)
idx_std = DynamicBiasedHNSW(data, M=16, ef_construction=32, random_state=42, build_on_init=True)

# Variant 2: Probabilistic Biased HNSW
idx_biased = DynamicBiasedHNSW(data, M=16, ef_construction=32, random_state=42, build_on_init=False)
idx_biased.rebuild_biased(frequencies, alpha=4.0)

# Variant 3: Biased + Heterogeneity Approach A (Min-Distance)
idx_hetero_md = DynamicBiasedHNSW(data, M=16, ef_construction=32, random_state=42, build_on_init=False)
idx_hetero_md.rebuild_heterogeneous_min_dist(frequencies, alpha=4.0, min_dist_percentile=25.0)

# Variant 4: Biased + Heterogeneity Approach B (Layer SSE Gain)
idx_hetero_sse = DynamicBiasedHNSW(data, M=16, ef_construction=32, random_state=42, build_on_init=False)
idx_hetero_sse.rebuild_heterogeneous_sse(frequencies, alpha=4.0, min_sse_gain_ratio=0.005)

variants = [
    ("Standard HNSW (Random)", idx_std),
    ("Biased HNSW (Probabilistic)", idx_biased),
    ("Biased + Hetero (Min-Dist)", idx_hetero_md),
    ("Biased + Hetero (SSE Gain)", idx_hetero_sse),
]

# 4. Benchmark evaluation
results = []
for name, idx_obj in variants:
    correct = 0
    total_hops = 0
    for q, target in zip(test_queries, test_gt):
        res = idx_obj.search(q, ef=32)
        if res['node'] == target:
            correct += 1
        total_hops += res['hops']

    recall = correct / len(test_queries)
    avg_hops = total_hops / len(test_queries)
    max_lvl = idx_obj.max_level
    top_layer_nodes = len(idx_obj.layers[max_lvl]) if max_lvl >= 0 else 0

    results.append({
        "Variant": name,
        "Recall@1 (%)": round(recall * 100, 2),
        "Avg Hops": round(avg_hops, 2),
        "Max Level": max_lvl,
        "Top Layer Node Count": top_layer_nodes
    })

df_results = pd.DataFrame(results)
display(df_results)


In [ ]:
# Visualizations: Performance Metrics and Upper-Layer Spatial Distribution

plt.figure(figsize=(14, 5))

# Plot 1: Recall@1 Comparison
plt.subplot(1, 2, 1)
colors = ['#7289da', '#f04747', '#43b581', '#faa61a']
bars = plt.bar(df_results['Variant'], df_results['Recall@1 (%)'], color=colors)
plt.title('1-NN Recall@1 Comparison (%)', fontsize=12, fontweight='bold')
plt.ylabel('Recall@1 (%)', fontsize=11)
plt.xticks(rotation=20, ha='right')
plt.ylim(0, 105)
for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2.0, yval + 1.5, f"{yval:.1f}%", ha='center', va='bottom', fontweight='bold')

# Plot 2: Average Search Hops Comparison
plt.subplot(1, 2, 2)
bars2 = plt.bar(df_results['Variant'], df_results['Avg Hops'], color=colors)
plt.title('Average Search Hops per Query', fontsize=12, fontweight='bold')
plt.ylabel('Avg Hops', fontsize=11)
plt.xticks(rotation=20, ha='right')
for bar in bars2:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2.0, yval + 0.5, f"{yval:.1f}", ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

# ---------------------------------------------------------
# PCA Spatial Distribution of Top-Layer Nodes
# ---------------------------------------------------------
print("Computing 2D PCA projection for top-layer node spatial analysis...")

pca = PCA(n_components=2, random_state=42)
data_2d = pca.fit_transform(data)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

vis_variants = [
    ("Biased HNSW (Probabilistic)", idx_biased, axes[0]),
    ("Biased + Hetero (Min-Dist)", idx_hetero_md, axes[1]),
    ("Biased + Hetero (SSE Gain)", idx_hetero_sse, axes[2]),
]

for name, idx_obj, ax in vis_variants:
    ax.scatter(data_2d[:, 0], data_2d[:, 1], c='lightgrey', s=10, alpha=0.4, label='All Data')
    
    # Identify top level (>0) nodes
    top_nodes = [n for l in range(1, len(idx_obj.layers)) for n in idx_obj.layers[l].keys()]

    if top_nodes:
        ax.scatter(data_2d[top_nodes, 0], data_2d[top_nodes, 1], c='crimson', s=40, edgecolors='black', label='Upper Layer Nodes')

    ax.set_title(f"{name}\n(Upper Layer Spatial Distribution)", fontsize=11, fontweight='bold')
    ax.legend(loc='upper right')

plt.tight_layout()
plt.show()
